In [1]:
!pip install alpaca-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 5.1 MB/s eta 0:00:00


In [2]:
import time
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from google.colab import userdata

In [3]:
API_KEY = userdata.get('ALPACA_KEY')
SECRET_KEY = userdata.get('ALPACA_SECRET')

# Inicializamos el cliente de Alpaca (paper=True para simulación)
client = TradingClient(api_key=API_KEY, secret_key=SECRET_KEY, paper=True)

In [4]:
# Tu presupuesto total a distribuir
PRESUPUESTO_TOTAL = 60.0  # Euros/Dólares

# Activos en los que queremos diversificar
activos = ["AAPL", "MSFT", "SPY"]

# Calculamos cuánto dinero va a cada activo (Fracciones por monto)
monto_por_activo = round(PRESUPUESTO_TOTAL / len(activos), 2)

print(f"🚀 Iniciando compra automatizada de fracciones...")
print(f"💰 Presupuesto Total: ${PRESUPUESTO_TOTAL}")
print(f"📊 Monto por cada activo: ${monto_por_activo}\n")

🚀 Iniciando compra automatizada de fracciones...
💰 Presupuesto Total: $60.0
📊 Monto por cada activo: $20.0



In [5]:
# EJECUCIÓN DE LAS ÓRDENES

for ticker in activos:
    try:
        # Creamos la orden usando 'notional' para indicar el dinero exacto
        market_order_data = MarketOrderRequest(
            symbol=ticker,
            notional=monto_por_activo, # <--- Aquí ocurre la magia de las fracciones
            side=OrderSide.BUY,
            time_in_force=TimeInForce.DAY
        )

        # Enviar orden al mercado
        order = client.submit_order(order_data=market_order_data)

        print(f"✅ Orden enviada con éxito para {ticker}:")
        print(f"   - Estado: {order.status}")
        print(f"   - Tipo: Compra fraccionada por ${monto_por_activo}")
        print("-" * 40)

    except Exception as e:
        print(f"❌ Error al comprar {ticker}: {e}")
        print("-" * 40)

✅ Orden enviada con éxito para AAPL:
   - Estado: OrderStatus.PENDING_NEW
   - Tipo: Compra fraccionada por $20.0
----------------------------------------
✅ Orden enviada con éxito para MSFT:
   - Estado: OrderStatus.PENDING_NEW
   - Tipo: Compra fraccionada por $20.0
----------------------------------------
✅ Orden enviada con éxito para SPY:
   - Estado: OrderStatus.PENDING_NEW
   - Tipo: Compra fraccionada por $20.0
----------------------------------------


In [6]:
# REVISIÓN DE TU PORTAFOLIO EN TIEMPO REAL

print("\n⏳ Esperando unos segundos para que se ejecuten las órdenes...")
time.sleep(3)

print("\n💼 Tu Portafolio Actualizado:")
try:
    posiciones = client.get_all_positions()
    if not posiciones:
        print("No tienes posiciones abiertas todavía.")
    for pos in posiciones:
        print(f"📈 {pos.symbol}: {pos.qty} acciones | Valor actual: ${pos.market_value}")
except Exception as e:
    print(f"No se pudieron recuperar las posiciones: {e}")


⏳ Esperando unos segundos para que se ejecuten las órdenes...

💼 Tu Portafolio Actualizado:
📈 AAPL: 0.064391914 acciones | Valor actual: $19.997656
📈 MSFT: 0.04850625 acciones | Valor actual: $19.987485
📈 SPY: 0.026641233 acciones | Valor actual: $19.993846


**El horario del mercado importa**: Las órdenes de fracciones en Alpaca se ejecutan como Market Orders (órdenes de mercado). Si ejecutas el script cuando el mercado de EE.UU. está cerrado (fuera del horario 9:30 AM - 4:00 PM EST), Alpaca dejará las órdenes "suspendidas" o pendientes hasta la próxima apertura.

**El parámetro clave**: Fíjate en la línea notional=monto_por_activo. Al usar notional en lugar de qty, Alpaca calcula automáticamente cuántas decimales de acción te corresponden según el precio del mercado en ese milisegundo.

### Ejecucion automatizada del script

In [ ]:
!pip install alpaca-py schedule

**Código con la regla de automatización mensual**

In [ ]:
import time
import datetime
import schedule
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

# =====================================================================
# 1. CONFIGURACIÓN Y CREDENCIALES
# =====================================================================
API_KEY = "TU_API_KEY_AQUÍ"
SECRET_KEY = "TU_SECRET_KEY_AQUÍ"

client = TradingClient(api_key=API_KEY, secret_key=SECRET_KEY, paper=True)

# Configuración de estrategia
PRESUPUESTO_TOTAL = 60.0
activos = ["AAPL", "MSFT", "SPY"]
monto_por_activo = round(PRESUPUESTO_TOTAL / len(activos), 2)

# =====================================================================
# 2. FUNCIÓN DE INVERSIÓN (La tarea que se repetirá)
# =====================================================================
def ejecutar_dca_mensual():
    print(f"\n🔔 [ALERTA] {datetime.datetime.now()}: Ejecutando orden mensual automatizada...")

    for ticker in activos:
        try:
            # Crear orden por monto monetario (fracciones)
            market_order_data = MarketOrderRequest(
                symbol=ticker,
                notional=monto_por_activo,
                side=OrderSide.BUY,
                time_in_force=TimeInForce.DAY
            )

            # Enviar orden
            order = client.submit_order(order_data=market_order_data)
            print(f"✅ Orden enviada: {ticker} por ${monto_por_activo} (Estado: {order.status})")

        except Exception as e:
            print(f"❌ Error al comprar {ticker}: {e}")

    # Esperar confirmación y mostrar portafolio
    time.sleep(5)
    mostrar_portafolio()

def mostrar_portafolio():
    print("\n💼 Estado actual de tu Portafolio:")
    try:
        posiciones = client.get_all_positions()
        if not posiciones:
            print("No hay posiciones abiertas.")
        for pos in posiciones:
            print(f"   📈 {pos.symbol}: {pos.qty} acciones | Valor: ${pos.market_value}")
    except Exception as e:
        print(f"Error al leer portafolio: {e}")

# =====================================================================
# 3. REGLA DE PROGRAMACIÓN (AUTOMATIZACIÓN)
# =====================================================================

# Opción A: Programar para el primer día de cada mes a las 10:00 AM
# Nota: La librería schedule no tiene un ".monthly", por lo que validamos el día del mes dentro de una regla diaria.
def verificar_y_ejecutar_calendario():
    hoy = datetime.date.today()
    # Si es el día 1 del mes, ejecuta la inversión
    if hoy.day == 1:
        ejecutar_dca_mensual()

# Programamos para que revise TODOS los días a las 10:00 AM si es día 1
schedule.every().day.at("10:00").do(verificar_y_ejecutar_calendario)

# ---------------------------------------------------------------------
# REGLA DE PRUEBA (Opcional):
# Descomenta la siguiente línea si quieres probar que el código funciona
# ejecutándose inmediatamente cada 10 segundos en lugar de esperar al mes.
# schedule.every(10).seconds.do(ejecutar_dca_mensual)
# ---------------------------------------------------------------------

print("🤖 Script de automatización iniciado.")
print("📅 Esperando al día 1 de cada mes a las 10:00 AM para invertir...")

# Bucle infinito para mantener el script escuchando el reloj
while True:
    schedule.run_pending()
    time.sleep(1)

### Prueba rápida

Como esperar al día 1 del próximo mes es mucho tiempo para saber si tu código tiene errores de sintaxis, ve a la sección 3. REGLA DE PRUEBA al final del script, borra el símbolo # de la línea schedule.every(10).seconds.do(ejecutar_dca_mensual) y ejecuta la celda. Verás cómo cada 10 segundos el programa intenta realizar las compras de las fracciones en tu cuenta de Alpaca.